# 02 – Download & Inspect Buildings

**Objective:** Download the confirmed public buildings GeoJSON from the Gdynia thermal viewer,
perform basic exploratory analysis, and save a raw GeoPackage to
`data/processed/buildings/buildings_raw.gpkg`.

In [ ]:
from __future__ import annotations

import geopandas as gpd
import pandas as pd
import requests
import yaml
from pathlib import Path

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"

with CONFIG_PATH.open() as fh:
    cfg = yaml.safe_load(fh)

BUILDINGS_URL = cfg["sources"]["buildings"]["url"]
RAW_GEOJSON   = PROJECT_ROOT / cfg["sources"]["buildings"]["local_path"]
RAW_GPKG      = PROJECT_ROOT / cfg["database"]["buildings_raw_path"]
PROJECT_CRS   = cfg["crs"]["project_crs"]

RAW_GEOJSON.parent.mkdir(parents=True, exist_ok=True)
RAW_GPKG.parent.mkdir(parents=True, exist_ok=True)

print("Buildings URL:", BUILDINGS_URL)

## 1. Download GeoJSON

In [ ]:
if not RAW_GEOJSON.exists():
    print("Downloading buildings GeoJSON …")
    resp = requests.get(BUILDINGS_URL, timeout=60)
    resp.raise_for_status()
    RAW_GEOJSON.write_bytes(resp.content)
    print(f"  Saved {len(resp.content):,} bytes to {RAW_GEOJSON}")
else:
    print(f"Using cached file: {RAW_GEOJSON}")

## 2. Load and explore

In [ ]:
gdf = gpd.read_file(RAW_GEOJSON)
print(f"Rows: {len(gdf):,}  |  CRS: {gdf.crs}")
gdf.head()

In [ ]:
# Basic statistics
print("Columns:", list(gdf.columns))
print("\nGeometry types:")
print(gdf.geometry.geom_type.value_counts())
print(f"\nBounding box (WGS 84):\n{gdf.total_bounds}")

## 3. Reproject and save raw GeoPackage

In [ ]:
gdf_proj = gdf.to_crs(PROJECT_CRS)
gdf_proj["area_m2"] = gdf_proj.geometry.area.round(2)

gdf_proj.to_file(RAW_GPKG, driver="GPKG", layer="buildings_raw")
print(f"Saved {len(gdf_proj):,} buildings to {RAW_GPKG}")
print(f"  area_m2 stats:\n{gdf_proj['area_m2'].describe().round(1)}")